# AutoML is the Future (embrace it don't run away)  

After repeated fails against AutoML-based techniques I have decided to succumb.  

Here is a guide on how to use AutoML.

**DISCLAIMER: NOT WELL FORMATED!**

In [24]:
# !pip install "autogluon.tabular[all]" --extra-index-url https://download.pytorch.org/whl/cpu
# !pip install autoviz --upgrade

## AutoViz - Easy data visualisation

We will use the salary dataset that killed me during the selection training camp, located at `https://drive.google.com/file/d/10Kct0vldQfTgz7zncIUk_ZC3He3aA6xh/view?usp=sharing` (train) and `https://drive.google.com/file/d/11eZvKmY3ULXMviulJvtH91v0KfTHyCk1/view?usp=sharing` (test).

In [5]:
!pip install gdown

In [8]:
import gdown
gdown.download("https://drive.google.com/uc?id=10Kct0vldQfTgz7zncIUk_ZC3He3aA6xh")
gdown.download("https://drive.google.com/uc?id=11eZvKmY3ULXMviulJvtH91v0KfTHyCk1")

Downloading...
From: https://drive.google.com/uc?id=10Kct0vldQfTgz7zncIUk_ZC3He3aA6xh
To: /home/walnit/Documents/NOAI/IOAI 2025/ML/ml_trainset.csv
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 186k/186k [00:00<00:00, 16.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=11eZvKmY3ULXMviulJvtH91v0KfTHyCk1
To: /home/walnit/Documents/NOAI/IOAI 2025/ML/ml_testset.csv
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18.0k/18.0k [00:00<00:00, 1.41MB/s]


'ml_testset.csv'

In [24]:
import pandas as pd

data = pd.read_csv("ml_trainset.csv")
data.head()

,Unnamed: 0,work_year,experience_level,employment_type,job_title,salary_currency,employee_residence,remote_ratio,company_location,company_size,salary_in_usd
0,0,2023,SE,FT,Principal Data Scientist,EUR,ES,100,ES,L,85847
1,1,2023,MI,CT,ML Engineer,USD,US,100,US,S,30000
2,2,2023,MI,CT,ML Engineer,USD,US,100,US,S,25500
3,3,2023,SE,FT,Data Scientist,USD,CA,100,CA,M,175000
4,4,2023,SE,FT,Data Scientist,USD,CA,100,CA,M,120000


This data is so hard to read. Let's use AutoViz.

In [25]:
from autoviz import AutoViz_Class

AV = AutoViz_Class()

In [3]:
dft = AV.AutoViz(
    "", # empty filename, library is a bit cooked
    dfte=data,
    depVar="salary_in_usd",
    verbose=1, # dont save too much stuff to disk,
    chart_format="server",
)

Shape of your Data Set loaded: (3379, 11)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  0
    Number of Integer-Categorical Columns =  1
    Number of String-Categorical Columns =  7
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  0
    Number of Numeric-Boolean Columns =  0
    Number of Discrete String Columns =  0
    Number of NLP String Columns =  0
    Number of Date Time Columns =  1
    Number of ID Columns =  1
    Number of Columns to Delete =  0
    10 Predictors classified...
        1 variable(s) removed since they were ID or low-information variables
        List of variables removed: ['Unnamed: 0']

################ Regression problem ################

scatterplots can be found in URL below:
Launching server at http://localhost:38105


distplots can be found in URL below:
Launching server at http://localhost:36213
distplots can be found in URL below:
Launching server at http://localhost:38761


kde_plots can be found in URL below:
Launching server at http://localhost:37891


violinplots can be found in URL below:
Launching server at http://localhost:37595


timeseries_plots can be found in URL below:
Launching server at http://localhost:41907


cat_var_plots can be found in URL below:
Launching server at http://localhost:43623
Time to run AutoViz (in seconds) = 7


From the plots, you can get a rough gauge of which variables are useful and which are not. Useful if you're doing manual data cleaning, but not so useful if you're using AutoGluon.  

However, we can also do some basic scaling and feature selection. First, lets automatically find data quality issues.

In [6]:
from autoviz import data_cleaning_suggestions

data_cleaning_suggestions(data)
print("") # remove double print

    All variables classified into correct types.


,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
Unnamed: 0,int64,0.000000,100,0.000000,3754.000000,Possible ID column: drop before modeling step.
work_year,int64,0.000000,0,2020.000000,2023.000000,Possible date-time colum: transform before modeling step.
experience_level,object,0.000000,0,,,No issue
employment_type,object,0.828648,0,,,"28 missing values. Impute them with mean, median, mode, or a constant value such as 123., 3 rare categories: ['PT', 'CT', 'FL']. Group them into a single category or drop the categories., Mixed dtypes: has 2 different data types: object, float,"
job_title,object,0.000000,2,,,83 rare categories: Too many to list. Group them into a single category or drop the categories.
salary_currency,object,0.000000,0,,,15 rare categories: Too many to list. Group them into a single category or drop the categories.
employee_residence,object,0.000000,2,,,69 rare categories: Too many to list. Group them into a single category or drop the categories.
remote_ratio,int64,0.000000,0,0.000000,100.000000,No issue
company_location,object,0.917431,1,,,"31 missing values. Impute them with mean, median, mode, or a constant value such as 123., 59 rare categories: Too many to list. Group them into a single category or drop the categories., Mixed dtypes: has 2 different data types: object, float,"
company_size,object,0.000000,0,,,No issue


Looks scary to fix, but luckily, we have FixDQ.

In [26]:
from autoviz import FixDQ

fixdq = FixDQ()

We can use Jupyter's autocomplete to see all the features it offers. We can also quickly, and automatically, fix the problems.

In [27]:
data = data.drop(["Unnamed: 0"], axis=1)
rare_cats = fixdq.group_rare_categories(data[["job_title", "salary_currency", "employee_residence", "company_location"]])
data.loc[:, ["job_title", "salary_currency", "employee_residence", "company_location"]] = rare_cats
data.loc[:, ['work_year', 'remote_ratio']] = data[['work_year', 'remote_ratio']].astype(str)
old_feats = ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary_currency', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']

categorical_feats = ['employment_type', 'job_title', 'salary_currency', 'employee_residence', 'company_location']
ordinal_feats = ['work_year', 'experience_level', 'remote_ratio', 'company_size']
data = pd.concat([data, pd.get_dummies(data[categorical_feats])], axis=1)
data = data.drop(categorical_feats, axis=1)
data.loc[:, "experience_level"] = pd.Categorical(data["experience_level"], ["EN", "MI", "SE", "EX"], ordered=True).codes
data.loc[:, "work_year"] = pd.Categorical(data["work_year"], ["2020", "2021", "2022", "2023"], ordered=True).codes
data.loc[:, "remote_ratio"] = pd.Categorical(data["remote_ratio"], ["0", "50", "100"], ordered=True).codes
data.loc[:, "company_size"] = pd.Categorical(data["company_size"], ["S", "M", "L"], ordered=True).codes
data.loc[:, "salary_in_usd"] = data["salary_in_usd"].clip(0, 295000)

data.head()

,work_year,experience_level,remote_ratio,company_size,salary_in_usd,employment_type_CT,employment_type_FL,employment_type_FT,employment_type_PT,job_title_Analytics Engineer,job_title_Applied Scientist,job_title_Data Analyst,job_title_Data Architect,job_title_Data Engineer,job_title_Data Science Manager,job_title_Data Scientist,job_title_Machine Learning Engineer,job_title_Rare,job_title_Research Engineer,job_title_Research Scientist,salary_currency_EUR,salary_currency_GBP,salary_currency_INR,salary_currency_Rare,salary_currency_USD,employee_residence_CA,employee_residence_DE,employee_residence_ES,employee_residence_FR,employee_residence_GB,employee_residence_IN,employee_residence_Rare,employee_residence_US,company_location_CA,company_location_DE,company_location_ES,company_location_GB,company_location_IN,company_location_Rare,company_location_US
0,3,2,2,2,85847,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False
1,3,1,2,0,30000,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True
2,3,1,2,0,25500,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True
3,3,2,2,1,175000,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False
4,3,2,2,1,120000,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False


In [31]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3379 entries, 0 to 3378
Data columns (total 40 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   work_year                            3379 non-null   object
 1   experience_level                     3379 non-null   object
 2   remote_ratio                         3379 non-null   object
 3   company_size                         3379 non-null   object
 4   salary_in_usd                        3379 non-null   int64 
 5   employment_type_CT                   3379 non-null   bool  
 6   employment_type_FL                   3379 non-null   bool  
 7   employment_type_FT                   3379 non-null   bool  
 8   employment_type_PT                   3379 non-null   bool  
 9   job_title_Analytics Engineer         3379 non-null   bool  
 10  job_title_Applied Scientist          3379 non-null   bool  
 11  job_title_Data Analyst               3379 n

Okay, let's pass this into the AutoML model!

## AutoGluon - Automatic ML

In [28]:
from autogluon.tabular import TabularDataset, TabularPredictor

AutoGluon techniques are used with a `TabularDataset`, which is a subclass of a normal DataFrame.

In [29]:
ag_df = TabularDataset(data)
ag_df.head()

,work_year,experience_level,remote_ratio,company_size,salary_in_usd,employment_type_CT,employment_type_FL,employment_type_FT,employment_type_PT,job_title_Analytics Engineer,job_title_Applied Scientist,job_title_Data Analyst,job_title_Data Architect,job_title_Data Engineer,job_title_Data Science Manager,job_title_Data Scientist,job_title_Machine Learning Engineer,job_title_Rare,job_title_Research Engineer,job_title_Research Scientist,salary_currency_EUR,salary_currency_GBP,salary_currency_INR,salary_currency_Rare,salary_currency_USD,employee_residence_CA,employee_residence_DE,employee_residence_ES,employee_residence_FR,employee_residence_GB,employee_residence_IN,employee_residence_Rare,employee_residence_US,company_location_CA,company_location_DE,company_location_ES,company_location_GB,company_location_IN,company_location_Rare,company_location_US
0,3,2,2,2,85847,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False
1,3,1,2,0,30000,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True
2,3,1,2,0,25500,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True
3,3,2,2,1,175000,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False
4,3,2,2,1,120000,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False


Training is simple.

In [30]:
predictor = TabularPredictor(label="salary_in_usd").fit(ag_df, presets="high", time_limit=300)

No path specified. Models will be saved in: "AutogluonModels/ag-20250625_160110"
Preset alias specified: 'high' maps to 'high_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.8
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu, 19 Jun 2025 14:41:19 +0000
CPU Count:          8
Memory Avail:       10.19 GB / 15.42 GB (66.1%)
Disk Space Avail:   34.79 GB / 224.81 GB (15.5%)
Presets specified: ['high']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk

(_ray_fit pid=47622) [1000]	valid_set's rmse: 41075.5


(_ray_fit pid=47622) 	Ran out of time, early stopping on iteration 3920. Best iteration is:
(_ray_fit pid=47622) 	[3330]	valid_set's rmse: 39715.3
(_ray_fit pid=47302) 	Ran out of time, stopping training early. (Stopping on epoch 5) [repeated 7x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


(_ray_fit pid=47618) [4000]	valid_set's rmse: 32776 [repeated 24x across cluster]


(_dystack pid=45707) 	-35547.0566	 = Validation score   (-root_mean_squared_error)
(_dystack pid=45707) 	13.45s	 = Training   runtime
(_dystack pid=45707) 	10.76s	 = Validation runtime
(_dystack pid=45707) Fitting model: WeightedEnsemble_L3 ... Training model for up to 70.79s of the -11.14s of remaining time.
(_dystack pid=45707) 	Ensemble Weights: {'LightGBMXT_BAG_L2': 0.895, 'CatBoost_BAG_L1': 0.105}
(_dystack pid=45707) 	-35377.6062	 = Validation score   (-root_mean_squared_error)
(_dystack pid=45707) 	0.04s	 = Training   runtime
(_dystack pid=45707) 	0.0s	 = Validation runtime
(_dystack pid=45707) AutoGluon training complete, total runtime = 82.61s ... Best model: WeightedEnsemble_L3 | Estimated inference throughput: 32.5 rows/s (376 batch size)
(_dystack pid=45707) Automatically performing refit_full as a post-fit operation (due to `.fit(..., refit_full=True)`
(_dystack pid=45707) Refitting models via `predictor.refit_full` using all of the data (combined train and validation)...


Fun fact, AutoGluon actually does not recommend us to do feature engineering, because it has in-built feature engineering. We can see what it does:

In [104]:
from autogluon.features.generators import AutoMLPipelineFeatureGenerator
auto_ml_feat_gen = AutoMLPipelineFeatureGenerator()
auto_ml_feat_gen.fit_transform(pd.read_csv("ml_testset.csv"))

Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    9990.43 MB
	Train Data (Original)  Memory Usage: 0.14 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('int', [])    : 3 | ['Unnamed: 0', 'work_year', 'remote_ratio']
		('object', []) : 7 | ['experience_level', 'employment_type', 'job_title', 'salary_currency', 'employee_residence', ...]
	Types of features in processed data (raw dtype, specia

,Unnamed: 0,work_year,remote_ratio,experience_level,employment_type,job_title,salary_currency,employee_residence,company_location,company_size
0,3208,2022,100,2,1,5,1,5,NaN,1
1,1692,2023,0,3,1,10,4,8,6,1
2,2031,2022,100,2,1,5,4,8,6,1
3,1651,2023,0,3,1,10,4,8,6,1
4,107,2023,0,3,1,2,4,8,6,1
5,259,2023,100,3,1,7,4,8,6,1
6,3058,2022,0,3,1,13,4,8,6,1
7,1950,2022,0,2,1,5,4,8,6,1
8,367,2023,0,2,1,16,2,4,4,1
9,1013,2023,0,3,1,12,4,8,6,1


The auto generation is... alright. But doing it ourselves makes more interpretable sense.  

Let's test the results. To do so, we need to format the test dataset in the way of our train dataset.

In [32]:
data = pd.read_csv("ml_testset.csv")
data = data.drop(["Unnamed: 0"], axis=1)
# AutoVis DQfix changes with the dataset...
# data.loc[:, ["job_title", "salary_currency", "employee_residence", "company_location"]] = fixdq.group_rare_categories(data[["job_title", "salary_currency", "employee_residence", "company_location"]])
data.loc[:, ['work_year', 'remote_ratio']] = data[['work_year', 'remote_ratio']].astype(str)
old_feats = ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary_currency', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']

categorical_feats = ['employment_type', 'job_title', 'salary_currency', 'employee_residence', 'company_location']
ordinal_feats = ['work_year', 'experience_level', 'remote_ratio', 'company_size']
# don't work on the un-DQed categorical variables
data = pd.concat([data, pd.get_dummies(data[["employment_type"]])], axis=1)
data = data.drop(["employment_type"], axis=1) 
data["experience_level"] = pd.Categorical(data["experience_level"], ["EN", "MI", "SE", "EX"], ordered=True).codes
data["work_year"] = pd.Categorical(data["work_year"], ["2020", "2021", "2022", "2023"], ordered=True).codes
data["remote_ratio"] = pd.Categorical(data["remote_ratio"], ["0", "50", "100"], ordered=True).codes
data["company_size"] = pd.Categorical(data["company_size"], ["S", "M", "L"], ordered=True).codes

for col in rare_cats.columns:
    data.loc[~data[col].isin(rare_cats[col].unique()), col] = "Rare"

data = pd.concat([data, pd.get_dummies(data[rare_cats.columns])], axis=1)

for col in ['employment_type_CT', 'employment_type_FL', 'employment_type_FT', 'employment_type_PT']:
    data.loc[:, col] = False

data = data.drop(['job_title', 'salary_currency', 'employee_residence', 'company_location'], axis=1) 

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 376 entries, 0 to 375
Data columns (total 39 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   work_year                            376 non-null    int8 
 1   experience_level                     376 non-null    int8 
 2   remote_ratio                         376 non-null    int8 
 3   company_size                         376 non-null    int8 
 4   employment_type_CT                   376 non-null    bool 
 5   employment_type_FL                   376 non-null    bool 
 6   employment_type_FT                   376 non-null    bool 
 7   employment_type_PT                   376 non-null    bool 
 8   job_title_Analytics Engineer         376 non-null    bool 
 9   job_title_Applied Scientist          376 non-null    bool 
 10  job_title_Data Analyst               376 non-null    bool 
 11  job_title_Data Architect             376 non-null    bool 

In [33]:
ag_test_df = TabularDataset(data)

y_pred = predictor.predict(ag_test_df)
y_pred.head()

0     46398.234375
1    140073.046875
2     98379.929688
3    140073.046875
4    139096.359375
Name: salary_in_usd, dtype: float32

In [34]:
from sklearn.metrics import r2_score

In [36]:
y_true = pd.read_csv("ml_testlabel.csv")
mean_squared_error(y_true["salary_in_usd"], y_pred)

2323972096.0

## What does a baseline do

In [4]:
data = pd.read_csv("ml_trainset.csv")

data = data.drop(["Unnamed: 0"], axis=1)
rare_cats = fixdq.group_rare_categories(data[["job_title", "salary_currency", "employee_residence", "company_location"]])
data.loc[:, ["job_title", "salary_currency", "employee_residence", "company_location"]] = rare_cats
data.loc[:, ['work_year', 'remote_ratio']] = data[['work_year', 'remote_ratio']].astype(str)
old_feats = ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary_currency', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']

categorical_feats = ['employment_type', 'job_title', 'salary_currency', 'employee_residence', 'company_location']
ordinal_feats = ['work_year', 'experience_level', 'remote_ratio', 'company_size']
data = pd.concat([data, pd.get_dummies(data[categorical_feats])], axis=1)
data = data.drop(categorical_feats, axis=1)
data["experience_level"] = pd.Categorical(data["experience_level"], ["EN", "MI", "SE", "EX"], ordered=True).codes
data["work_year"] = pd.Categorical(data["work_year"], ["2020", "2021", "2022", "2023"], ordered=True).codes
data["remote_ratio"] = pd.Categorical(data["remote_ratio"], ["0", "50", "100"], ordered=True).codes
data["company_size"] = pd.Categorical(data["company_size"], ["S", "M", "L"], ordered=True).codes
data.loc[:, "salary_in_usd"] = data["salary_in_usd"].clip(0, 295000)

X = data.drop("salary_in_usd", axis=1)
y = data["salary_in_usd"]

In [5]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

model = XGBRegressor()

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 5, 10],
    "learning_rate": [1e-3, 1e-2, 1e-1]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    verbose=1,
    scoring="neg_mean_squared_error"
)

grid_search.fit(X, y)
grid_search.best_params_

Fitting 5 folds for each of 27 candidates, totalling 135 fits


{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100}

In [6]:
best_model = grid_search.best_estimator_

In [19]:
data = pd.read_csv("ml_testset.csv")
data = data.drop(["Unnamed: 0"], axis=1)
# AutoVis DQfix changes with the dataset...
# data.loc[:, ["job_title", "salary_currency", "employee_residence", "company_location"]] = fixdq.group_rare_categories(data[["job_title", "salary_currency", "employee_residence", "company_location"]])
data.loc[:, ['work_year', 'remote_ratio']] = data[['work_year', 'remote_ratio']].astype(str)
old_feats = ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary_currency', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']

categorical_feats = ['employment_type', 'job_title', 'salary_currency', 'employee_residence', 'company_location']
ordinal_feats = ['work_year', 'experience_level', 'remote_ratio', 'company_size']
# don't work on the un-DQed categorical variables
data = pd.concat([data, pd.get_dummies(data[["employment_type"]])], axis=1)
data = data.drop(["employment_type"], axis=1) 
data["experience_level"] = pd.Categorical(data["experience_level"], ["EN", "MI", "SE", "EX"], ordered=True).codes
data["work_year"] = pd.Categorical(data["work_year"], ["2020", "2021", "2022", "2023"], ordered=True).codes
data["remote_ratio"] = pd.Categorical(data["remote_ratio"], ["0", "50", "100"], ordered=True).codes
data["company_size"] = pd.Categorical(data["company_size"], ["S", "M", "L"], ordered=True).codes

for col in rare_cats.columns:
    data.loc[~data[col].isin(rare_cats[col].unique()), col] = "Rare"

data = pd.concat([data, pd.get_dummies(data[rare_cats.columns])], axis=1)

for col in ['employment_type_CT', 'employment_type_FL', 'employment_type_FT', 'employment_type_PT']:
    data.loc[:, col] = False

data = data.drop(['job_title', 'salary_currency', 'employee_residence', 'company_location'], axis=1) 

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 376 entries, 0 to 375
Data columns (total 39 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   work_year                            376 non-null    int8 
 1   experience_level                     376 non-null    int8 
 2   remote_ratio                         376 non-null    int8 
 3   company_size                         376 non-null    int8 
 4   employment_type_CT                   376 non-null    bool 
 5   employment_type_FL                   376 non-null    bool 
 6   employment_type_FT                   376 non-null    bool 
 7   employment_type_PT                   376 non-null    bool 
 8   job_title_Analytics Engineer         376 non-null    bool 
 9   job_title_Applied Scientist          376 non-null    bool 
 10  job_title_Data Analyst               376 non-null    bool 
 11  job_title_Data Architect             376 non-null    bool 

In [21]:
y_pred = best_model.predict(data)

In [23]:
y_true = pd.read_csv("ml_testlabel.csv")
mean_squared_error(y_true["salary_in_usd"], y_pred)

2164775936.0

In [147]:
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBRegressor
import np

skf = StratifiedKFold(n_splits=5)
best_mse = np.inf
for i, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_val, y_train, y_val = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]
    model = XGBRegressor()
    
    
    